In [109]:
from util.benchmark_parser import WorkerBenchmarkParser
from util.encoding import WorkerEncoding

parser = WorkerBenchmarkParser()
encoding = parser.parse_benchmark(r'instances/fjssp-w/3_DPpaulli_1_workers.fjs')

Total operations for job 1: 15
	Options for operation 1 (index 0): 1
		Option 1: Machine 1
			Worker options: 7
				(Worker 1, duration 45)
				(Worker 2, duration 38)
				(Worker 3, duration 42)
				(Worker 4, duration 40)
				(Worker 5, duration 43)
				(Worker 6, duration 42)
				(Worker 7, duration 44)
	Options for operation 2 (index 1): 1
		Option 1: Machine 3
			Worker options: 2
				(Worker 2, duration 62)
				(Worker 5, duration 66)
	Options for operation 3 (index 2): 1
		Option 1: Machine 5
			Worker options: 6
				(Worker 1, duration 22)
				(Worker 2, duration 21)
				(Worker 3, duration 21)
				(Worker 4, duration 24)
				(Worker 5, duration 21)
				(Worker 6, duration 21)
	Options for operation 4 (index 3): 1
		Option 1: Machine 4
			Worker options: 6
				(Worker 1, duration 16)
				(Worker 2, duration 15)
				(Worker 3, duration 17)
				(Worker 4, duration 15)
				(Worker 5, duration 15)
				(Worker 7, duration 17)
	Options for operation 5 (index 4): 3
		Option 1: Machine

In [110]:
from enum import Enum
from dataclasses import dataclass
import random

class Strategy(Enum):
    PLUS = 1
    COMMA = 2

@dataclass
class Operation:
    machine_index: int
    worker_index: int
    job_index: int
    operation_index: int
    duration: int
    offset: int

class Candidate:
    def __init__(self, schedule: list[list[Operation]]) -> None:
        self.schedule = schedule
        self.time = max(machine[-1].duration + machine[-1].offset for machine in schedule)

    def __repr__(self) -> str:
        # {[[(task.job_index, task.task_index, task.duration, task.offset) for task in machine] for machine in self.schedule]}
        return f"{self.time}"

def create_candidate(schedule, machine_ready_times, operation_ready_times, next_op_by_job, last_operation_by_job, active_jobs, encoding: WorkerEncoding) -> Candidate:
    # populate schedule for each machine
    while active_jobs:
        selected_job = random.choice(active_jobs)
        selected_op = next_op_by_job[selected_job]

        # select random viable machine and worker
        machines = encoding.get_machines_for_operation(selected_op)
        selected_machine = random.choice(machines)
        workers = encoding.get_workers_for_operation_on_machine(selected_op, selected_machine)
        selected_worker = random.choice(workers)

        offset = max(operation_ready_times[selected_job], machine_ready_times[selected_machine])
        duration = encoding.durations()[selected_op, selected_machine, selected_worker]
        op_info = Operation(selected_machine, selected_worker, selected_job, selected_op, duration, offset)
        schedule[selected_machine].append(op_info)

        operation_ready_times[selected_job] = duration + offset # Question: is it correct that a later operation cannot be started until all earlier operations have been finished?
        machine_ready_times[selected_machine] = duration + offset

        # check if this was the last task for this job
        next_op_by_job[selected_job] += 1
        if next_op_by_job[selected_job] > last_operation_by_job[selected_job]:
            active_jobs.remove(selected_job)
    
    return Candidate(schedule)

def get_initial_candidates(m, num_jobs, num_machines, last_operation_by_job, encoding: WorkerEncoding):
    candidates = list[Candidate]()
    for _ in range(m):
        # create a schedule for each machine
        schedule = [list[Operation]() for _ in range(num_machines)]

        # for each machine, store when it becomes available again
        machine_ready_times = [0] * num_machines
        operation_ready_times = [0] * num_jobs

        # for each job, store the index of the next operation to schedule
        next_operation_by_job = [0] * num_jobs
        operation_index = 0
        for i in range(num_jobs):
            ops_for_current_job = encoding.get_operations_for_job(i)
            next_operation_by_job[i] = operation_index
            operation_index += ops_for_current_job

        # jobs that still have tasks left
        active_jobs = [index for index in range(num_jobs)]
        candidate = create_candidate(schedule, machine_ready_times, operation_ready_times, next_operation_by_job, last_operation_by_job, active_jobs, encoding)
        candidates.append(candidate)
    return candidates

def mutate(parents: list[Candidate], l, num_jobs, num_machines, last_operation_by_job, encoding: WorkerEncoding) -> list[Candidate]:
    children_per_parent = l / len(parents)
    children = list[Candidate]()
    for parent in parents:
        for _ in range(int(children_per_parent)):
            # select random machine and operation to split at
            random_machine_idx = random.randint(0, num_machines - 1)
            operations_for_machine = len(parent.schedule[random_machine_idx])
            random_operation_idx = random.randint(0, operations_for_machine - 1)
            chosen_op = parent.schedule[random_machine_idx][random_operation_idx]
            schedule = [list[Operation]() for _ in range(num_machines)]
            
            # copy parent schedule to child
            for i in range(num_machines):
                operations_for_machine = len(parent.schedule[i])
                for j in range(operations_for_machine):
                    if parent.schedule[i][j].offset <= chosen_op.offset:
                        schedule[i].append(parent.schedule[i][j])
            
            # reconstruct state from the partial schedule
            machine_ready_times = [0] * num_machines
            operation_ready_times = [0] * num_jobs
            next_operation_by_job = [0] * num_jobs
            
            # scan through copied tasks to rebuild state
            for i, machine_tasks in enumerate(schedule):
                if machine_tasks:
                    last_task = machine_tasks[-1]
                    machine_ready_times[i] = last_task.offset + last_task.duration
                
            # find the highest task_index for each job
            for job_index in range(num_jobs):
                max_operation_index = -1
                for machine_tasks in schedule:
                    for task in machine_tasks:
                        if task.job_index == job_index and task.operation_index > max_operation_index:
                            max_operation_index = task.operation_index
                            operation_ready_times[job_index] = task.offset + task.duration
                next_operation_by_job[job_index] = max_operation_index + 1
            
            # only jobs with remaining tasks are active
            active_jobs = [job_index for job_index in range(num_jobs) if next_operation_by_job[job_index] < last_operation_by_job[job_index]]
            
            candidate = create_candidate(schedule, machine_ready_times, operation_ready_times, next_operation_by_job, last_operation_by_job, active_jobs, encoding)
            children.append(candidate)
    return children

def solve(strategy: Strategy, m, l, max_generations, encoding: WorkerEncoding) -> Candidate:
    num_jobs = encoding.n_jobs()
    num_machines = encoding.n_machines()

    last_operation_by_job = [0] * num_jobs
    operation_index = 0
    for i in range(num_jobs):
        ops_for_current_job = encoding.get_operations_for_job(i)
        operation_index += ops_for_current_job
        last_operation_by_job[i] = operation_index - 1

    parents = get_initial_candidates(m, num_jobs, num_machines, last_operation_by_job, encoding)
    print(parents)
    offsprings = list[Candidate]()
    for _ in range(max_generations):
        offsprings = mutate(parents, l, num_jobs, num_machines, last_operation_by_job, encoding)
        if strategy == Strategy.PLUS:
            offsprings.extend(parents)
            offsprings.sort(key=lambda x: x.time) 
            parents = []
            for i in range(m):
                parents.append(offsprings[i])
            print(parents)
        elif strategy == Strategy.COMMA:
            offsprings.sort(key=lambda x: x.time) 
            parents = []
            for i in range(m):
                parents.append(offsprings[i])
            print(parents)
    parents.sort(key=lambda x: x.time) 
    return parents[0]

In [111]:
s = solve(Strategy.PLUS, 10, 50, 50, encoding)

[4583, 4572, 4565, 4701, 4006, 4297, 4594, 4752, 5070, 4623]
[3961, 4006, 4056, 4139, 4147, 4205, 4209, 4262, 4273, 4275]
[3836, 3961, 4006, 4008, 4038, 4040, 4046, 4056, 4076, 4091]
[3803, 3836, 3961, 3979, 3991, 3994, 4006, 4008, 4025, 4037]
[3612, 3721, 3803, 3810, 3836, 3874, 3876, 3961, 3973, 3975]
[3612, 3692, 3701, 3721, 3773, 3784, 3803, 3810, 3821, 3833]
[3612, 3620, 3674, 3692, 3701, 3710, 3717, 3721, 3721, 3753]
[3569, 3612, 3620, 3623, 3644, 3674, 3677, 3687, 3692, 3692]
[3518, 3526, 3551, 3569, 3571, 3579, 3580, 3612, 3620, 3623]
[3444, 3518, 3526, 3547, 3551, 3551, 3569, 3571, 3574, 3576]
[3444, 3458, 3460, 3479, 3490, 3503, 3518, 3526, 3529, 3547]
[3401, 3443, 3444, 3458, 3460, 3479, 3490, 3503, 3518, 3524]
[3401, 3408, 3410, 3443, 3444, 3458, 3460, 3479, 3490, 3497]
[3401, 3408, 3410, 3440, 3443, 3444, 3458, 3458, 3460, 3460]
[3401, 3408, 3410, 3440, 3443, 3444, 3449, 3458, 3458, 3458]
[3401, 3408, 3410, 3439, 3440, 3443, 3444, 3449, 3456, 3458]
[3365, 3401, 3408, 3410,